# AWS 与 SageMaker 实战指南

本笔记整合原 `aws.ipynb` 与 `sagemaker.ipynb`,演示如何在 AWS 云上创建 GPU 实例、部署 Jupyter 环境,并使用 Amazon SageMaker 完成训练与推理。所有命令基于官方 CLI,可在本地或 CloudShell 中执行。

## 1. 账号与权限准备

1. 注册 AWS 账号并开启 MFA。
2. 在 IAM 中创建具备以下权限的用户/角色:
   - `AmazonEC2FullAccess`
   - `AmazonS3FullAccess`
   - `AmazonSageMakerFullAccess`
3. 本地配置凭证:

```bash
aws configure
# 依次输入 Access Key、Secret Key、默认区域(如 ap-northeast-1)、输出格式(json)
```

## 2. 在 EC2 上创建深度学习环境

### 2.1 选择 AMI 与实例

- 推荐使用 `Deep Learning AMI GPU PyTorch` 或 `DLAMI Base`。
- 实例类型示例:
  - 入门: `g4dn.xlarge` (T4, 16 GB)
  - 进阶: `g5.2xlarge` (A10G, 24 GB)
  - 大模型: `p3.2xlarge` (V100) / `p4d.24xlarge` (A100)

### 2.2 启动实例

```bash
aws ec2 run-instances   --image-id ami-xxxxxxxx   --count 1   --instance-type g5.2xlarge   --key-name my-key   --security-group-ids sg-xxxxxxx   --subnet-id subnet-xxxxxx   --block-device-mappings '[{"DeviceName":"/dev/sda1","Ebs":{"VolumeSize":200}}]'
```

> 建议将根卷扩大到 200 GB,方便存放数据与模型。

### 2.3 配置环境

```bash
ssh -i my-key.pem ubuntu@ec2-xx-xx-xx-xx.compute.amazonaws.com
```

连接后执行:

```bash
sudo apt update && sudo apt upgrade -y
conda activate pytorch  # DLAMI 默认提供
pip install --upgrade pip
pip install notebook matplotlib pandas scikit-learn
jupyter lab --no-browser --port 8888
```

若需端口转发:

```bash
ssh -i my-key.pem -L 8888:localhost:8888 ubuntu@ec2-...
```

### 2.4 保存快照

训练结束后可创建 AMI 保存配置:

```bash
aws ec2 create-image --instance-id i-xxxx --name d2l-snapshot
```

## 3. 使用 Amazon S3 管理数据

```bash
aws s3 mb s3://d2l-bucket-123
aws s3 sync ./data s3://d2l-bucket-123/data
```

在 Notebook 中可使用 `boto3` 读取:

```python
import boto3
s3 = boto3.client('s3')
obj = s3.get_object(Bucket='d2l-bucket-123', Key='data/train.csv')
```

## 4. Amazon SageMaker 工作流程

### 4.1 创建 Notebook Instance

1. 在 SageMaker 控制台选择 *Notebook instances* → *Create*。
2. 推荐实例: `ml.t3.medium`(入门) 或 `ml.g4dn.xlarge`(GPU)。
3. 选择 IAM 角色,勾选“访问 S3”。
4. 启动后点击 *Open JupyterLab* 即可进入开发环境。

### 4.2 使用 SageMaker Python SDK 训练模型

以下示例演示如何在托管训练作业中运行 PyTorch 脚本。`train.py` 需上传到 S3。



In [ ]:
import sagemaker
from sagemaker.pytorch import PyTorch

role = sagemaker.get_execution_role()
sess = sagemaker.Session()

estimator = PyTorch(
    entry_point='train.py',
    role=role,
    instance_count=1,
    instance_type='ml.g5.xlarge',
    framework_version='2.1',
    py_version='py310',
    hyperparameters={
        'epochs': 5,
        'batch_size': 64,
        'lr': 1e-3,
    },
)

estimator.fit({'training': 's3://d2l-bucket-123/data/'})



### 4.3 部署推理服务

训练完成后可一键部署:

```python
predictor = estimator.deploy(initial_instance_count=1, instance_type='ml.m5.large')
response = predictor.predict({'inputs': [0.5, 0.3, 0.1]})
```

若无需托管服务,可使用 `estimator.latest_training_job.download_model()` 下载模型自行部署。

### 4.4 关停资源

```python
predictor.delete_endpoint()
```

> 别忘了在控制台终止 Notebook Instance 与训练作业,以免产生额外费用。

## 5. 成本控制建议

- 使用 `ml.t3.medium` Notebook + Spot 训练组合,费用较低。
- 训练脚本中保存 checkpoint 至 S3,便于中断后恢复。
- 为 S3 存储设置生命周期策略,过期文件自动转入 Glacier。

## 6. 最佳实践清单

- [ ] 使用 IAM Role 最小权限原则
- [ ] 训练脚本读取 `SM_MODEL_DIR`、`SM_CHANNEL_TRAINING` 环境变量
- [ ] 日志写入 `print()` 或 `logging`,自动同步到 CloudWatch
- [ ] 使用 `sagemaker.experiments` 管理多个实验
- [ ] 训练完毕删除 Endpoint、Notebook Instance

---
借助本篇,你可以在 AWS 平台上完成从环境搭建到自动化训练的全流程。若需要了解社区贡献与资源,继续阅读 `07_d2l社区与资源.ipynb`。

